# Autonomous Agent Prediction Beta First Pass

Audience: Kaggle competitors using this repository to prepare agent-style submissions.

Prerequisites: basic Python, pandas, NumPy, and the idea of a Kaggle submission file.

Learning goals:

- Understand why this competition submits `submission.zip` instead of a static `submission.csv`.
- Inspect the real sample schema when it is available locally.
- Build a tiny leakage-safe binary-classification baseline on synthetic data.
- Validate the structure of an agent config folder before packaging.
- Package a zip with `agent.yaml` at the archive root.

Outline:

1. Load official workspace references.
2. Create a synthetic tabular mini-competition.
3. Train a simple probability baseline and compute ROC AUC.
4. Build and validate a minimal agent-config folder.
5. Package the final `submission.zip` shape.


## Official Facts

The official evaluation page says the submitted artifact is a zip archive named `submission.zip` with `agent.yaml` at the archive root. The agent then runs in a Kaggle CPU sandbox, creates prediction files during the session, calls `submit_predictions`, and selects final submissions.

The metric is ROC AUC, so predictions should be continuous probabilities in `[0, 1]`. Hard class labels throw away ranking information.


In [ ]:
from pathlib import Path
import os
import shutil
import zipfile

import numpy as np
import pandas as pd


def find_repo_file(relative_path):
    relative_path = Path(relative_path)
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    return None

sample_path = find_repo_file('competitions/autonomous-agent-prediction-beta/references/data-samples/sample_submission.csv')
data_md_path = find_repo_file('competitions/autonomous-agent-prediction-beta/references/data-samples/DATA.md')
print('sample_submission found:', bool(sample_path), sample_path)
print('DATA.md found:', bool(data_md_path), data_md_path)

if sample_path:
    sample = pd.read_csv(sample_path)
    print('sample shape:', sample.shape)
    print(sample.head(3))


## Synthetic Mini-Competition

This fallback creates a small binary-classification task with the same practical shape as `train_01`: a `row_id`, mixed numeric/categorical features, and a binary `target` present only in train.


In [ ]:
rng = np.random.default_rng(7)

def make_frame(n, include_target):
    x0 = rng.normal(size=n)
    x1 = rng.normal(size=n)
    x2 = rng.normal(size=n)
    cat = rng.choice(['a', 'b', 'c'], size=n, p=[0.55, 0.30, 0.15])
    logits = 1.4 * x0 - 0.9 * x1 + 0.6 * (cat == 'b') - 0.5 * (cat == 'c') + 0.25 * x2
    probs = 1 / (1 + np.exp(-logits))
    frame = pd.DataFrame({
        'row_id': [f'id_{i:05d}' for i in range(n)],
        'feature_0': cat,
        'feature_1': x0,
        'feature_2': x1,
        'feature_3': x2,
    })
    if include_target:
        frame['target'] = rng.binomial(1, probs)
    return frame

train_df = make_frame(900, include_target=True)
test_df = make_frame(300, include_target=False)
sample_submission = pd.DataFrame({'row_id': test_df['row_id'], 'target': 0.5})
print(train_df.head())
print('train shape:', train_df.shape, 'test shape:', test_df.shape)
print('target mean:', round(train_df['target'].mean(), 4))


## Transparent Probability Baseline

This dependency-light logistic baseline is not meant to beat tree ensembles. The transferable habits are: split validation rows before looking at scores, encode categoricals without target leakage, predict probabilities, and evaluate with ROC AUC.


In [ ]:
def roc_auc_score_np(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(y_score) + 1)
    pos = y_true == 1
    n_pos = int(pos.sum())
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return float('nan')
    return (ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def make_design(train_part, valid_or_test):
    train_x = pd.get_dummies(train_part.drop(columns=['row_id', 'target']), dummy_na=True)
    other_x = pd.get_dummies(valid_or_test.drop(columns=['row_id'], errors='ignore'), dummy_na=True)
    other_x = other_x.reindex(columns=train_x.columns, fill_value=0)
    means = train_x.mean(axis=0)
    stds = train_x.std(axis=0).replace(0, 1)
    return ((train_x - means) / stds).to_numpy(float), ((other_x - means) / stds).to_numpy(float)


def fit_logistic_gd(x, y, steps=700, lr=0.08, l2=0.01):
    x = np.c_[np.ones(len(x)), x]
    w = np.zeros(x.shape[1])
    y = np.asarray(y, dtype=float)
    for _ in range(steps):
        p = 1 / (1 + np.exp(-(x @ w)))
        grad = (x.T @ (p - y)) / len(y)
        grad[1:] += l2 * w[1:]
        w -= lr * grad
    return w


def predict_logistic(w, x):
    x = np.c_[np.ones(len(x)), x]
    return 1 / (1 + np.exp(-(x @ w)))

valid_idx = rng.choice(train_df.index, size=int(0.25 * len(train_df)), replace=False)
valid_mask = train_df.index.isin(valid_idx)
tr = train_df.loc[~valid_mask].reset_index(drop=True)
va = train_df.loc[valid_mask].reset_index(drop=True)

x_tr, x_va = make_design(tr, va)
w = fit_logistic_gd(x_tr, tr['target'])
valid_pred = predict_logistic(w, x_va)
auc = roc_auc_score_np(va['target'], valid_pred)
print('holdout ROC AUC:', round(float(auc), 4))

x_full, x_test = make_design(train_df, test_df)
w_full = fit_logistic_gd(x_full, train_df['target'])
test_pred = predict_logistic(w_full, x_test)
submission = sample_submission.copy()
submission['target'] = np.clip(test_pred, 0, 1)
print(submission.head())
print('prediction range:', float(submission['target'].min()), float(submission['target'].max()))


## Minimal Agent Folder

A real upload must follow the official demo/sample schema. This local folder only teaches structural habits: root `agent.yaml`, internal prompt includes, skill manifests, and clean zip contents.


In [ ]:
work_dir = Path('/tmp/autonomous_agent_prediction_tutorial')
submission_root = work_dir / 'demo_submission'
if submission_root.exists():
    shutil.rmtree(submission_root)
(submission_root / 'prompts').mkdir(parents=True)
(submission_root / 'skills' / 'feature-engineer').mkdir(parents=True)

agent_yaml = """name: tutorial_agent
model: gemini-3.5-flash
instruction: !include prompts/system.md
tools:
  - run_command
  - submit_predictions
  - select_submission
  - get_status
skills:
  - skills/feature-engineer
generate_content_config:
  temperature: 0.1
  max_output_tokens: 4096
"""
(submission_root / 'agent.yaml').write_text(agent_yaml, encoding='utf-8')

system_prompt = """You are an autonomous Kaggle agent.
First create a valid probability submission.csv, then call submit_predictions.
If unsure, submit the current valid file instead of continuing to plan.
"""
(submission_root / 'prompts' / 'system.md').write_text(system_prompt, encoding='utf-8')

skill_md = """---
name: feature-engineer
description: Create a fallback-safe tabular binary-classification submission.
---
# Feature Engineer

Discover train/test/sample files, infer ID and target columns, and always write a valid submission.csv.
"""
(submission_root / 'skills' / 'feature-engineer' / 'SKILL.md').write_text(skill_md, encoding='utf-8')

print('created', submission_root)
print(sorted(str(p.relative_to(submission_root)) for p in submission_root.rglob('*') if p.is_file()))


In [ ]:
import yaml

ALLOWED_TOOLS = {'run_command', 'submit_predictions', 'select_submission', 'get_status', 'write_file', 'edit_file'}

class IncludeTag:
    def __init__(self, path):
        self.path = path


def include_constructor(loader, node):
    return IncludeTag(loader.construct_scalar(node))

yaml.SafeLoader.add_constructor('!include', include_constructor)


def validate_agent_folder(root):
    root = Path(root).resolve()
    issues = []
    config_path = root / 'agent.yaml'
    if not config_path.exists():
        return ['missing agent.yaml']
    cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
    for key in ['name', 'model', 'instruction', 'tools']:
        if key not in cfg:
            issues.append(f'missing required key: {key}')
    instruction = cfg.get('instruction')
    if isinstance(instruction, IncludeTag):
        target = (root / instruction.path).resolve()
        if not str(target).startswith(str(root)):
            issues.append('instruction include escapes submission root')
        elif not target.exists():
            issues.append(f'missing include target: {instruction.path}')
    for tool in cfg.get('tools', []):
        if isinstance(tool, str) and tool not in ALLOWED_TOOLS:
            issues.append(f'unknown tool: {tool}')
    for skill_dir in cfg.get('skills', []):
        if not (root / skill_dir / 'SKILL.md').exists():
            issues.append(f'missing SKILL.md for {skill_dir}')
    return issues

issues = validate_agent_folder(submission_root)
print('issues:', issues if issues else 'none')
assert not issues


In [ ]:
zip_path = work_dir / 'submission.zip'
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in submission_root.rglob('*'):
        if path.is_file() and '__pycache__' not in path.parts and not path.name.endswith('.pyc'):
            zf.write(path, path.relative_to(submission_root))

with zipfile.ZipFile(zip_path) as zf:
    names = sorted(zf.namelist())

print('zip:', zip_path)
print('\n'.join(names))
assert 'agent.yaml' in names
assert all(not name.endswith('.pyc') for name in names)


## Exercise

Change the synthetic data generator so one category is rare and strongly predictive. Re-run the baseline and compare ROC AUC.

Answer scaffold:

```python
# In make_frame, adjust the category probabilities and the category coefficient.
# Then re-run the baseline cell and compare the holdout ROC AUC.
```


## Pitfalls And Extensions

Common mistake: validating the source folder but uploading an old zip. Always reopen the final `submission.zip` and check its contents.

Extension: replace the dependency-light logistic baseline with the schema-agnostic feature-engineer skill described in `references/public-notebooks/NOTEBOOK_SUMMARY.md`, then score it across all practice folders with a replay harness.
